# MBDD2025 — simple YOLO11s retraining

Train a candidate defect detector from the Kaggle dataset [`asadibehnam/mbdd2025-uav-building-defect-detection`](https://www.kaggle.com/datasets/asadibehnam/mbdd2025-uav-building-defect-detection), a mirror of [MBDD2025 on Zenodo](https://doi.org/10.5281/zenodo.15622584). The Kaggle page states CC BY 4.0; credit the original dataset authors when redistributing.

**Classes (fixed order):** `crack`, `leakage`, `abscission`, `corrosion`, `bulge`.

This notebook keeps preprocessing simple: pair images/XML by filename stem, convert VOC boxes to YOLO labels, clip boxes extending beyond image bounds, split reproducibly 70/20/10, train pretrained YOLO11s, and evaluate once on the held-out test set. It reports how many boxes were clipped or dropped. The split is image-level because the dataset has no verified building/flight ID, so related views may occur across splits.

The exported weights are a **candidate only**. They do not replace or deploy the current service model.

In [ ]:
%pip -q install "ultralytics==8.3.55" scikit-learn pandas pillow

In [ ]:
from collections import Counter
from pathlib import Path
import hashlib
import json
import math
import platform
import random
import shutil
import xml.etree.ElementTree as ET

import pandas as pd
import torch
import ultralytics
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42
CLASS_NAMES = ["crack", "leakage", "abscission", "corrosion", "bulge"]
CLASS_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}
DATASET_REF = "asadibehnam/mbdd2025-uav-building-defect-detection"
IMAGE_SIZE = 640
EPOCHS = 100
BATCH_SIZE = 16

input_root = Path("/kaggle/input")
dataset_roots = [path for path in input_root.iterdir() if path.is_dir()]
dataset_root = next(
    (root for root in dataset_roots
     if (root / "JPEGImages").is_dir() and (root / "Annotations").is_dir()),
    None,
)
if dataset_root is None:
    raise FileNotFoundError("Attach the asadibehnam MBDD2025 dataset in Kaggle Input.")
image_dir = dataset_root / "JPEGImages"
annotation_dir = dataset_root / "Annotations"

work_dir = Path("/kaggle/working")
yolo_dir = work_dir / "mbdd2025-yolo"
export_dir = work_dir / "export"
audit_dir = work_dir / "dataset-audit"

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before training."
print("Dataset:", DATASET_REF)
print("Images:", image_dir)
print("Annotations:", annotation_dir)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Parse the VOC XML files, pair by shared release filename stem, and convert
# coordinates. Some source boxes extend slightly beyond image edges; clipping to
# the declared image dimensions is standard for object detection. Invalid boxes
# that have no visible area after clipping are dropped and counted.
image_extensions = {".jpg", ".jpeg", ".png"}
image_paths = sorted(
    (path for path in image_dir.iterdir()
     if path.is_file() and path.suffix.lower() in image_extensions),
    key=lambda path: path.name.casefold(),
)
xml_by_stem = {path.stem.casefold(): path for path in annotation_dir.glob("*.xml")}
if len(xml_by_stem) != len(list(annotation_dir.glob("*.xml"))):
    raise RuntimeError("Duplicate annotation filename stems found.")
missing_xml = [path.name for path in image_paths if path.stem.casefold() not in xml_by_stem]
if missing_xml:
    raise RuntimeError(f"Images are missing XML annotations (first 10): {missing_xml[:10]}")

records = []
object_counts = Counter()
clipped_boxes = 0
dropped_boxes = 0
empty_images = 0
for image_path in image_paths:
    xml_path = xml_by_stem[image_path.stem.casefold()]
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    if size is None:
        raise ValueError(f"Missing image dimensions in {xml_path.name}")
    width = int(float(size.findtext("width", "0")))
    height = int(float(size.findtext("height", "0")))
    if width <= 0 or height <= 0:
        raise ValueError(f"Invalid dimensions in {xml_path.name}: {width}x{height}")

    yolo_boxes = []
    labels = Counter()
    for object_index, obj in enumerate(root.findall("object")):
        label = obj.findtext("name", "").strip().casefold()
        if label not in CLASS_TO_ID:
            raise ValueError(f"Unknown class {label!r} in {xml_path.name}")
        bounds = obj.find("bndbox")
        if bounds is None:
            raise ValueError(f"Missing box {object_index} in {xml_path.name}")
        x_min = float(bounds.findtext("xmin", "nan"))
        y_min = float(bounds.findtext("ymin", "nan"))
        x_max = float(bounds.findtext("xmax", "nan"))
        y_max = float(bounds.findtext("ymax", "nan"))
        if not all(math.isfinite(value) for value in (x_min, y_min, x_max, y_max)):
            raise ValueError(f"Non-finite box {object_index} in {xml_path.name}")

        original = (x_min, y_min, x_max, y_max)
        x_min = min(max(x_min, 0.0), float(width))
        y_min = min(max(y_min, 0.0), float(height))
        x_max = min(max(x_max, 0.0), float(width))
        y_max = min(max(y_max, 0.0), float(height))
        if original != (x_min, y_min, x_max, y_max):
            clipped_boxes += 1
        if x_max <= x_min or y_max <= y_min:
            dropped_boxes += 1
            continue

        x_center = ((x_min + x_max) / 2) / width
        y_center = ((y_min + y_max) / 2) / height
        box_width = (x_max - x_min) / width
        box_height = (y_max - y_min) / height
        yolo_boxes.append(
            f"{CLASS_TO_ID[label]} {x_center:.8f} {y_center:.8f} "
            f"{box_width:.8f} {box_height:.8f}"
        )
        labels[label] += 1
        object_counts[label] += 1

    if not labels:
        empty_images += 1
        dominant_class = "background"
    else:
        dominant_class = min(CLASS_NAMES, key=lambda name: (-labels[name], CLASS_TO_ID[name]))
    records.append({
        "image": image_path,
        "xml": xml_path,
        "labels": yolo_boxes,
        "class_counts": labels,
        "dominant_class": dominant_class,
    })

print(f"Paired images: {len(records)}")
print(f"Boxes clipped to image bounds: {clipped_boxes}")
print(f"Boxes dropped after clipping (no visible area): {dropped_boxes}")
print(f"Images with no remaining boxes: {empty_images}")
print("Object counts:", {name: object_counts[name] for name in CLASS_NAMES})
if len(records) != 14_471:
    print(f"WARNING: Kaggle describes 14,471 images; found {len(records)}.")

In [ ]:
# Reproducible approximately 70/20/10 image-level split. Stratify by each
# image's dominant class to reduce class imbalance between the splits.
random.seed(SEED)
dominant_classes = [record["dominant_class"] for record in records]
try:
    train_val, test = train_test_split(
        records,
        test_size=0.10,
        random_state=SEED,
        stratify=dominant_classes,
    )
    train, val = train_test_split(
        train_val,
        test_size=2 / 9,  # 20% of total after holding out the 10% test split
        random_state=SEED + 1,
        stratify=[record["dominant_class"] for record in train_val],
    )
    print("Split is stratified by dominant class.")
except ValueError as error:
    # Rare classes can make stratification impossible; a plain random split still
    # produces a usable, reproducible dataset.
    print("Stratified split unavailable, falling back to random split:", error)
    train_val, test = train_test_split(records, test_size=0.10, random_state=SEED)
    train, val = train_test_split(train_val, test_size=2 / 9, random_state=SEED + 1)
splits = {"train": train, "val": val, "test": test}

if yolo_dir.exists():
    shutil.rmtree(yolo_dir)
for split_name, split_records in splits.items():
    images_out = yolo_dir / "images" / split_name
    labels_out = yolo_dir / "labels" / split_name
    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)
    for record in split_records:
        (images_out / record["image"].name).symlink_to(record["image"].resolve())
        # Empty label files retain negative/background images.
        (labels_out / f"{record['image'].stem}.txt").write_text(
            "\n".join(record["labels"]), encoding="utf-8"
        )

names = "\n".join(f"  {index}: {name}" for index, name in enumerate(CLASS_NAMES))
data_yaml = work_dir / "mbdd2025.yaml"
data_yaml.write_text(
    f"path: {yolo_dir}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n{names}\n",
    encoding="utf-8",
)

class_rows = []
manifest_rows = []
for split_name, split_records in splits.items():
    for class_name in CLASS_NAMES:
        class_rows.append({
            "split": split_name,
            "class": class_name,
            "images": sum(record["class_counts"][class_name] > 0 for record in split_records),
            "instances": sum(record["class_counts"][class_name] for record in split_records),
        })
    for record in split_records:
        manifest_rows.append({"split": split_name, "image": record["image"].name, "xml": record["xml"].name})

audit_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(class_rows).to_csv(audit_dir / "class_distribution.csv", index=False)
pd.DataFrame(manifest_rows).to_csv(audit_dir / "split_manifest.csv", index=False)
print("Split image counts:", {name: len(rows) for name, rows in splits.items()})
print(pd.DataFrame(class_rows).to_string(index=False))
print("YOLO config:", data_yaml)

In [ ]:
# Visual sanity check: show one annotated image for each class before training.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

samples = []
for class_name in CLASS_NAMES:
    sample = next((record for record in records if record["class_counts"][class_name]), None)
    if sample:
        samples.append((class_name, sample))
fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 4))
if len(samples) == 1:
    axes = [axes]
for axis, (class_name, record) in zip(axes, samples):
    with Image.open(record["image"]) as image:
        axis.imshow(image.convert("RGB"))
    root = ET.parse(record["xml"]).getroot()
    size = root.find("size")
    width = int(float(size.findtext("width")))
    height = int(float(size.findtext("height")))
    labels = []
    for obj in root.findall("object"):
        label = obj.findtext("name", "").strip().casefold()
        bounds = obj.find("bndbox")
        x0 = min(max(float(bounds.findtext("xmin")), 0), width)
        y0 = min(max(float(bounds.findtext("ymin")), 0), height)
        x1 = min(max(float(bounds.findtext("xmax")), 0), width)
        y1 = min(max(float(bounds.findtext("ymax")), 0), height)
        if x1 > x0 and y1 > y0:
            axis.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor="lime", linewidth=2))
            labels.append(label)
    axis.set_title(f"{class_name}: {record['image'].name}")
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Train from Ultralytics COCO-pretrained YOLO11s. Keep this result as a candidate;
# do not copy it into the running service automatically.
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=20,
    device=0,
    workers=4,
    optimizer="AdamW",
    lr0=0.002,
    cos_lr=True,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
    seed=SEED,
    deterministic=True,
    plots=True,
    project=str(work_dir / "runs"),
    name="mbdd2025-yolo11s-candidate",
    exist_ok=False,
)
run_dir = Path(model.trainer.save_dir)
best_weights = run_dir / "weights" / "best.pt"
if not best_weights.is_file():
    raise FileNotFoundError(f"Training did not produce {best_weights}")
print("Training run:", run_dir)
print("Candidate checkpoint:", best_weights)

In [ ]:
# Evaluate only once on the held-out test split after selecting best.pt by validation.
test_model = YOLO(str(best_weights))
expected_names = {index: name for index, name in enumerate(CLASS_NAMES)}
actual_names = {int(index): str(name) for index, name in test_model.names.items()}
if actual_names != expected_names:
    raise RuntimeError(f"Checkpoint class mapping mismatch: {actual_names}")

metrics = test_model.val(
    data=str(data_yaml),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    plots=True,
    project=str(work_dir / "test-evaluation"),
    name="held-out-test",
    exist_ok=False,
)
per_class = {}
for index, class_id in enumerate(metrics.box.ap_class_index):
    class_name = actual_names[int(class_id)]
    per_class[class_name] = {
        "precision": float(metrics.box.p[index]),
        "recall": float(metrics.box.r[index]),
        "map50": float(metrics.box.ap50[index]),
        "map50_95": float(metrics.box.ap[index]),
    }
for class_name in CLASS_NAMES:
    per_class.setdefault(class_name, None)

test_report = {
    "dataset": DATASET_REF,
    "test_images": len(test),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
    "per_class": per_class,
    "clipped_boxes": clipped_boxes,
    "dropped_boxes": dropped_boxes,
}
print(json.dumps(test_report, indent=2))

In [ ]:
# Package candidate weights and compact evidence files for retrieval from Kaggle.
export_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_weights, export_dir / "best.pt")
shutil.copy2(run_dir / "results.csv", export_dir / "training_results.csv")
if (run_dir / "args.yaml").is_file():
    shutil.copy2(run_dir / "args.yaml", export_dir / "training_args.yaml")
shutil.copy2(audit_dir / "class_distribution.csv", export_dir / "class_distribution.csv")
shutil.copy2(audit_dir / "split_manifest.csv", export_dir / "split_manifest.csv")
(export_dir / "test_metrics.json").write_text(json.dumps(test_report, indent=2), encoding="utf-8")

weight_hash = hashlib.sha256((export_dir / "best.pt").read_bytes()).hexdigest()
provenance = {
    "dataset": DATASET_REF,
    "dataset_url": "https://www.kaggle.com/datasets/" + DATASET_REF,
    "source_doi": "10.5281/zenodo.15622584",
    "class_order": CLASS_NAMES,
    "seed": SEED,
    "split": "sorted image/XML pairs; dominant-class stratified; image-level 70/20/10",
    "training": {
        "base": "yolo11s.pt (COCO pretrained)",
        "epochs": EPOCHS,
        "imgsz": IMAGE_SIZE,
        "batch": BATCH_SIZE,
        "optimizer": "AdamW",
        "lr0": 0.002,
        "clipped_boxes": clipped_boxes,
        "dropped_boxes": dropped_boxes,
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "ultralytics": ultralytics.__version__,
        "gpu": torch.cuda.get_device_name(0),
    },
    "weight_sha256": weight_hash,
    "test_metrics": test_report,
    "warning": "Candidate only; not deployed.",
}
(export_dir / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
for plot in (run_dir / "results.png", run_dir / "confusion_matrix.png", run_dir / "PR_curve.png"):
    if plot.is_file():
        shutil.copy2(plot, export_dir / plot.name)
print("Candidate package:")
for path in sorted(export_dir.iterdir()):
    print(path.name, path.stat().st_size, "bytes")